# Week 08 — Home exercise 3: The NASDAQ in real terms

**Solution proposal.**

Monthly resampling, then a merge of the annual series with consumer prices, so that fifty years of
the index can be read in constant dollars instead of the dollars of the day.

In [1]:
import pandas as pd

nasdaq = pd.read_csv("../data/NASDAQ.csv")
nasdaq["Date"] = pd.to_datetime(nasdaq["Date"], format="%Y-%m-%d")

prices = nasdaq.set_index("Date").sort_index()["NASDAQ"]

print(prices.dtype, "|", len(prices), "trading days")

float64 | 13842 trading days


## 1. The three worst months

In [2]:
monthly = prices.resample("ME").last()
monthly_return = monthly.pct_change() * 100

monthly_return.sort_values().head(3).round(1)

Date
1987-10-31   -27.2
2000-11-30   -22.9
2001-02-28   -22.4
Name: NASDAQ, dtype: float64

**October 1987, down 27.2%** — "Black Monday", 19 October 1987, when markets around the world fell
more than 20% in a single day. It remains the largest one-day percentage fall in modern stock market
history, and nearly forty years later it is still the worst month in this file.

The other two, November 2000 and February 2001, are the dot-com crash, visible month by month. Note
that neither 2008 nor March 2020 makes the list: the financial crisis was a long slide rather than one
terrible month, and the pandemic crash was spread over February and March and made up again by the
end of May.

## 2. Bringing in prices

`resample` leaves the dates in the index. `reset_index()` brings them back as a column, and `.dt.year`
gives the key the price file can be matched on.

In [3]:
annual = prices.resample("YE").last().reset_index()
annual["year"] = annual["Date"].dt.year

annual.head(3)

,Date,NASDAQ,year
0,1971-12-31,114.1,1971
1,1972-12-31,133.7,1972
2,1973-12-31,92.2,1973


In [4]:
fred = pd.read_excel("../data/FRED_annual.xlsx")

print(fred.shape)
print(fred["Year"].min(), "to", fred["Year"].max())
fred.head(3)

(70, 6)
1954 to 2023


,Year,GDP,CPI,UNRATE,FEDFUNDS,INFLATION
0,1954,2877.7,26.9,5.6,1.0,NaN
1,1955,3083.0,26.8,4.4,1.8,-0.371747
2,1956,3148.8,27.2,4.1,2.7,1.492537


One row per year on both sides, so the merge is **one-to-one**. That is the first time this week the
shape is genuinely one-to-one, and it is worth saying so: if either file had a year twice, the merge
would refuse to run.

The two files call the key `year` and `Year`, so `left_on` and `right_on` name each side.

In [5]:
real = annual.merge(
    fred[["Year", "CPI"]],
    left_on="year",
    right_on="Year",
    how="left",
    validate="one_to_one",
    indicator=True,
)

print(real["_merge"].value_counts())
real[real["_merge"] == "left_only"][["year", "NASDAQ", "CPI"]]

_merge
both          53
left_only      2
right_only     0
Name: count, dtype: int64


,year,NASDAQ,CPI
53,2024,19310.8,NaN
54,2025,23419.1,NaN


**53 years matched and two did not: 2024 and 2025.** The price file ends in 2023, the index two years
later. Nothing is wrong with either file; they were simply published to different dates.

**A left join is the right choice.** It keeps every year of the index, and leaves CPI missing where
it is unknown — so the nominal figures for 2024 and 2025 survive, and the real ones are honestly
`NaN`. An inner join would have quietly shortened the index by two years, and the `_merge` count is
the only thing that would have told you.

## 3. Real returns

Dividing by the price level and multiplying by 1999's price level expresses every year's index in
**1999 dollars**. 1999 is chosen so that the real and nominal series meet in the year question 4 is
about; any year would give the same returns.

In [6]:
cpi_1999 = real.loc[real["year"] == 1999, "CPI"].iloc[0]

real["real_1999"] = real["NASDAQ"] / real["CPI"] * cpi_1999
real["nominal_return"] = real["NASDAQ"].pct_change() * 100
real["real_return"] = real["real_1999"].pct_change() * 100

print("CPI in 1999:", cpi_1999)
real[["year", "NASDAQ", "CPI", "real_1999", "nominal_return", "real_return"]].head(4).round(1)

CPI in 1999: 166.6


,year,NASDAQ,CPI,real_1999,nominal_return,real_return
0,1971,114.1,40.5,469.4,NaN,NaN
1,1972,133.7,41.8,532.9,17.2,13.5
2,1973,92.2,44.4,346.0,-31.0,-35.1
3,1974,59.8,49.3,202.1,-35.1,-41.6


In [7]:
columns = ["year", "nominal_return", "real_return"]

print("worst five, real:")
print(real.sort_values("real_return")[columns].head(5).round(1))
print()
print("worst five, nominal:")
print(real.sort_values("nominal_return")[columns].head(5).round(1))

worst five, real:
    year  nominal_return  real_return
37  2008           -40.5        -42.8
3   1974           -35.1        -41.6
29  2000           -39.3        -41.3
51  2022           -33.1        -38.0
2   1973           -31.0        -35.1

worst five, nominal:
    year  nominal_return  real_return
37  2008           -40.5        -42.8
29  2000           -39.3        -41.3
3   1974           -35.1        -41.6
51  2022           -33.1        -38.0
31  2002           -31.5        -32.6


The same four disasters lead both lists — 2008, 2000, 1974 and 2022 — but the real ranking moves
**1974 up to second**, and brings **1973 into the five** in place of 2002. Consumer prices rose 6%
in 1973 and 11% in 1974, so the real loss was several points worse than the nominal one.

Which decade suffers most is clearer with a summary by decade — the decade key from the lecture,
applied to a column this time.

In [8]:
real["decade"] = real["year"] // 10 * 10

by_decade = real.groupby("decade")[["nominal_return", "real_return"]].mean()
by_decade["inflation_gap"] = by_decade["nominal_return"] - by_decade["real_return"]

by_decade.round(1)

,nominal_return,real_return,inflation_gap
decade,,,
1970,6.8,-0.6,7.4
1980,12.6,6.7,5.9
1990,27.6,24.0,3.6
2000,-0.9,-3.2,2.3
2010,15.6,13.6,2.0
2020,20.9,14.4,6.5


**The 1970s.** In nominal terms they look like a modest decade, up 6.8% in an average year. In real
terms they are the worst decade in the file, **down 0.6% in an average year** — every bit of the
nominal gain was inflation.

That is the real point of the exercise. In nominal terms, the 2000s are the only bad decade the
NASDAQ has had. In real terms, there were two, and the first one is invisible unless you bring in a
second file.

In [9]:
real_levels = real.set_index("year")["real_1999"]

peak_1972 = real_levels.loc[1972]
back_above = real_levels[(real_levels.index > 1972) & (real_levels > peak_1972)]

print(f"real level, end of 1972: {peak_1972:6.1f}")
print(f"lowest after that:       {real_levels.loc[1973:1982].min():6.1f}")
print("first year back above it:", back_above.index.min())

real level, end of 1972:  532.9
lowest after that:        202.1
first year back above it: 1988


Measured from the end of 1972, the index lost more than half of its real value within two years, and
did not get back to its 1972 level in real terms until **1988** — sixteen years. The 1970s had a lost
decade too, and in real terms it was longer than a decade.

## 4. The lost decade, again

In [10]:
levels = real.set_index("year")[["NASDAQ", "real_1999"]]

for column in ["NASDAQ", "real_1999"]:
    start = levels.loc[1999, column]
    end = levels.loc[2009, column]
    later = levels[(levels.index > 1999) & (levels[column] > start)]
    print(f"{column:10}  1999: {start:7.1f}   2009: {end:7.1f}   change: {(end / start - 1) * 100:6.1f}%"
          f"   first year back above 1999: {later.index.min()}")

NASDAQ      1999:  4069.3   2009:  2269.1   change:  -44.2%   first year back above 1999: 2013
real_1999   1999:  4069.3   2009:  1761.6   change:  -56.7%   first year back above 1999: 2017


**In nominal terms, the index ended 2009 down 44.2% on 1999, and was back above its 1999 level in
2013. In real terms it was down 56.7%, and did not get back until 2017** — seventeen years after the
peak, rather than thirteen.

The two series meet in 1999 by construction, which is why the starting levels are identical.

### What that does to "shares go up over time"

The claim is not false, and it is not a promise about any particular decade. Over the full period the
index went from about 100 to more than 23 000, which is an enormous return even after inflation. But
it got there through **two** periods in which somebody who bought at the top waited sixteen or
seventeen years to get their purchasing power back — and the first of those periods does not show up
at all until the second file is brought in.

Somebody who needed their money in 1980 or in 2009 got the decade, not the average.

Two honest caveats on this specific evidence:

- **The NASDAQ is not the market.** It is unusually concentrated in technology companies, which is
  exactly why the dot-com crash hit it so hard. A broader index fell much less over the same decade.
- **An index is not an investment.** It ignores dividends, which add meaningfully to long-run returns,
  and it ignores the fees and taxes of actually holding anything.

### Things worth noticing

- **A real series is a merge.** Anything measured in money over time needs a price index beside it
  before it can be compared across years, and that price index almost always comes from another file.
- **`indicator=True` is worth it even when you expect a perfect match.** Here it found the two years
  the files do not share, and it made the choice between a left and an inner join a decision rather
  than an accident.
- **`validate="one_to_one"` is the right claim for two annual series.** If either file ever gained a
  duplicated year — a revision appended rather than replacing — the merge would stop instead of
  doubling a row.

### What this notebook does NOT do

- **It mixes two kinds of annual figure.** The index is the closing level on the last trading day of
  the year; FRED's annual CPI is an average over the year. A stricter version would use December's CPI,
  which is a monthly series this file does not have.
- **The decade averages are simple means of yearly returns**, not the growth a holder actually
  experienced over the decade. A year down 50% followed by a year up 50% averages zero and leaves you
  25% poorer. The lost-decade calculations in question 4 use levels, and are the honest version.
- **Monthly returns are nominal only.** CPI here is annual, so question 1 cannot be done in real terms
  — though over a single month, inflation is too small to change the ranking.
- **The returns are index returns**, not investor returns: no dividends, no fees, no tax.